# Kolokvijum II — gde je šta odrađeno

Ovaj notebook prolazi kroz **tekst zadatka rečenicu po rečenicu** i za svaku pokazuje tačno gde je
u kodu ispunjena i čime se to dokazuje.

Zahtevi su numerisani **Z1–Z15**, a isti brojevi stoje kao komentari u kodu — pa se na odbrani može
doslovno pokazati prstom na red.

*(Za objašnjenje **zašto** neki mehanizam radi kako radi — slike, teorija — to je `k2_teorija_zadatka.ipynb`.
Ovaj notebook odgovara samo na pitanje **gde**.)*

## Pregled

| | Zahtev | Gde |
| --- | --- | --- |
| Z1 | konstruktori za Workflow i Task | `const Task = function`, `const Workflow = function` |
| Z2 | Task ima naziv, definiciju i execute | tri reda u `Task` |
| Z3 | definicija se izvršava pozivom execute | `return definicija(...)` |
| Z4 | rezultat execute = rezultat poziva definicije | isti red |
| Z5 | execute prima proizvoljan broj argumenata | `(...argumenti)` i `definicija(...argumenti)` |
| Z6 | Workflow ima naziv, autor, spisak, execute | `this.naziv`, `this.autor`, geter `zadaci`, `this.execute` |
| Z7 | Workflow je iterabilan, daje Task objekte | `this[Symbol.iterator]` |
| Z8 | map, flat, flatMap, filter | četiri metode |
| Z9 | ulaz svakog sledećeg = rezultat prethodnog | `ulaz = [await ...]` u petlji |
| Z10 | prvi task dobija sve prosleđene argumente | `let ulaz = argumenti` |
| Z11 | povratna vrednost = rezultat poslednjeg | `return ulaz[0]` |
| Z12 | oba execute su asinhrona | `async` na obe metode |
| Z13 | proizvoljna kompozicija transdjusera | `transduce` i `primeni` + `poredjaj` |
| Z14 | transdjuser za ispis svakog koraka | `saIspisom` |
| Z15 | demonstracija sa više objekata | poslednja ćelija |

---
# Pasus 1, prvi deo — `Task`

> „Napraviti konstruktore za objekte tipa **Workflow** i **Task**. Svaki Task opisan je **nazivom,
> definicijom i metodom execute**. **Definicija predstavlja funkciju koja se izvršava pozivom funkcije
> execute.** Rezultat poziva metode execute nad Task objektom je **rezultat poziva funkcije date u
> atributu definicija**. Metoda execute može primiti **proizvoljan broj argumenata** koji prosleđuje
> funkciji definisanoj u atributu definicija."

| | Zahtev | Red u kodu | Kako je ispunjen |
| --- | --- | --- | --- |
| **Z1** | konstruktor za Task | `const Task = function (naziv, definicija)` | `function` konstruktor, koristi se sa `new` |
| **Z2** | naziv, definicija, execute | `this.naziv`, `this.definicija`, `this.execute` | tri polja na instanci |
| **Z3** | definicija se izvršava pozivom execute | `return definicija(...argumenti);` | telo `execute`-a ne radi ništa drugo |
| **Z4** | rezultat execute = rezultat definicije | isti red | `return` prosleđuje vrednost bez izmene |
| **Z5** | proizvoljan broj argumenata | `(...argumenti)` pa `definicija(...argumenti)` | rest **pakuje**, spread **raspakuje** |

Z3 i Z4 su isti red gledan iz dva ugla: „šta se poziva" i „šta se vraća".

In [ ]:
const Task = function (naziv, definicija) {             // Z1
  this.naziv = naziv;                                   // Z2
  this.definicija = definicija;                         // Z2

  this.execute = async function (...argumenti) {        // Z2, Z5 (rest), Z12 (async)
    return definicija(...argumenti);                    // Z3, Z4, Z5 (spread)
  };
};

// ── dokazi ────────────────────────────────────────────────────────
const zbir = new Task("zbir", (niz) => niz.reduce((a, x) => a + x, 0));

console.log("Z2 tri polja:", Object.keys(zbir));
console.log("Z3 definicija je funkcija:", typeof zbir.definicija);
console.log("Z4 execute = poziv definicije:", await zbir.execute([1, 2, 3]), "===", zbir.definicija([1, 2, 3]));

const koliko = new Task("koliko", (...sve) => sve.length);
console.log("Z5 jedan argument:", await koliko.execute(1), "| tri:", await koliko.execute(1, 2, 3));

---
# Pasus 1, drugi deo — `Workflow`

> „**Workflow sadrži podatke naziv, autor, spisak Task objekata i metodu execute.** Workflow je
> **iterabilni objekat, iterisanjem kroz workflow dobijaju se pojedinačni Task objekti.** Za Workflow
> definisati operacije **map, flat, flatMap i filter**."

| | Zahtev | Red u kodu | Kako je ispunjen |
| --- | --- | --- | --- |
| **Z1** | konstruktor za Workflow | `const Workflow = function (naziv, autor, zadaci = [])` | isti obrazac kao `Task` |
| **Z6** | naziv, autor, spisak, execute | `this.naziv`, `this.autor`, geter `zadaci`, `this.execute` | spisak ide preko getera koji vraća kopiju |
| **Z7** | iterabilan, daje Task objekte | `this[Symbol.iterator] = function* () { yield* zadaci; }` | generator pod standardnim ključem |
| **Z8** | map, flat, flatMap, filter | četiri metode, svaka `return new Workflow(...)` | vraćaju **nov** Workflow, pa se mogu nizati |

**Zašto geter za `zadaci`.** Da je `this.zadaci = zadaci`, spolja bi se moglo `wf.zadaci.push(...)` i
promeniti sam workflow. Geter vraća kopiju, pa spisak ostaje netaknut.

**Zašto `flat` traži pomoćnu funkciju.** `Workflow` može da sadrži drugi `Workflow`. Pošto je
`Workflow` iterabilan (Z7), izravnavanje ne pita za tip nego za sposobnost:
`typeof z[Symbol.iterator] === "function"`.

In [ ]:
const izravnaj = function (zadaci) {                          // pomocna za Z8
  return zadaci.flatMap((z) =>
    typeof z[Symbol.iterator] === "function" ? [...z] : [z],  // pita za sposobnost, ne za tip
  );
};

const Workflow = function (naziv, autor, zadaci = []) {       // Z1
  this.naziv = naziv;                                         // Z6
  this.autor = autor;                                         // Z6

  Object.defineProperty(this, "zadaci", {                     // Z6 — spisak, preko kopije
    get() { return [...zadaci]; },
    enumerable: true,
  });

  this[Symbol.iterator] = function* () { yield* zadaci; };    // Z7

  this.map = function (f) { return new Workflow(naziv, autor, zadaci.map(f)); };              // Z8
  this.filter = function (p) { return new Workflow(naziv, autor, zadaci.filter(p)); };        // Z8
  this.flat = function () { return new Workflow(naziv, autor, izravnaj(zadaci)); };           // Z8
  this.flatMap = function (f) { return new Workflow(naziv, autor, izravnaj(zadaci.map(f))); };// Z8

  this.execute = async function (...argumenti) {              // Z6, Z12 (async)
    let ulaz = argumenti;                                     // Z10 — prvi dobija SVE
    for (const zadatak of zadaci) {
      ulaz = [await zadatak.execute(...ulaz)];                // Z9 — izlaz jednog je ulaz sledeceg
    }
    return ulaz[0];                                           // Z11 — rezultat poslednjeg
  };

  this.transduce = function (transdjuser, kraj = sakupi, pocetno = []) {   // Z13
    return zadaci.reduce(transdjuser(kraj), pocetno);
  };

  this.primeni = function (...transdjuseri) {                              // Z13
    return new Workflow(naziv, autor, zadaci.reduce(poredjaj(...transdjuseri)(sakupi), []));
  };
};

In [ ]:
// ── podaci za sve dalje dokaze ────────────────────────────────────
const parsiraj = new Task("parsiraj", (t) => t.split(",").map(Number));
const bezNula = new Task("bezNula", (n) => n.filter((x) => x !== 0));
const formatiraj = new Task("formatiraj", (x) => "ukupno: " + x);
const cekaj = new Task("cekaj", async (x) => {
  await new Promise((k) => setTimeout(k, 20));
  return x;
});
const saberiTri = new Task("saberiTri", (a, b, c) => a + b + c);
const puta10 = new Task("puta10", (x) => x * 10);

const priprema = new Workflow("priprema", "Blagoje", [parsiraj, bezNula]);
const racun = new Workflow("racun", "Blagoje", [cekaj, zbir]);
const glavni = new Workflow("glavni", "Blagoje", [priprema, racun, formatiraj]);
const ravan = glavni.flat();

// ── dokazi ────────────────────────────────────────────────────────
console.log("Z6 podaci:", priprema.naziv, "|", priprema.autor, "|", priprema.zadaci.length, "zadatka |", typeof priprema.execute);
priprema.zadaci.push(formatiraj);
console.log("Z6 spisak zasticen:", [...priprema].length, "— geter vraca kopiju");

console.log("\nZ7 for...of daje Task objekte:");
for (const z of priprema) console.log("   ", z.naziv, "| ima definiciju:", typeof z.definicija);
console.log("Z7 spread:", [...priprema].map((z) => z.naziv));

console.log("\nZ8 flat:   ", [...ravan].map((z) => z.naziv), "— raspakovan ugnjezden Workflow");
console.log("Z8 filter: ", [...ravan.filter((z) => z.naziv !== "cekaj")].map((z) => z.naziv));
console.log("Z8 map:    ", [...ravan.map((z) => new Task(z.naziv.toUpperCase(), z.definicija))].map((z) => z.naziv));
console.log("Z8 flatMap:", [...priprema.flatMap((z) => new Workflow("p", "B", [z, new Task("kroz", (x) => x)]))].map((z) => z.naziv));
console.log("Z8 sve vracaju Workflow, pa se nizu:", [...ravan.filter((z) => z.naziv !== "cekaj").map((z) => z)].length);

---
# Pasus 2 — `execute`

> „Metoda execute izvršava sve zadate Task objekte tako što **kao ulaz u execute metodu Task objekta
> dostavlja rezultat prethodno izvršenog Task objekta**. Za ulaz u metodu za **prvi task objekat uzimaju
> se svi prosleđeni argumenti** iz metode execute u Workflow objektu. **Povratna vrednost metode je
> rezultat izvršavanja poslednjeg** Task objekta iz niza. Execute metode iz Workflow i Task objekta se
> **izvršavaju asinhrono**."

Ceo pasus je pokriven sa **pet redova**:

```js
this.execute = async function (...argumenti) {        //      Z12
  let ulaz = argumenti;                               // Z10
  for (const zadatak of zadaci) {
    ulaz = [await zadatak.execute(...ulaz)];          // Z9   Z12
  }
  return ulaz[0];                                     // Z11
};
```

| | Zahtev | Red | Kako |
| --- | --- | --- | --- |
| **Z9** | ulaz sledećeg = rezultat prethodnog | `ulaz = [await ...]` | rezultat postaje ulaz naredne runde |
| **Z10** | prvi dobija sve argumente | `let ulaz = argumenti` | početna vrednost je **ceo niz** argumenata |
| **Z11** | vraća se rezultat poslednjeg | `return ulaz[0]` | posle petlje `ulaz` drži samo poslednji rezultat |
| **Z12** | asinhrono | `async` + `await` | `execute` uvek vraća `Promise` |

**Zašto je `ulaz` niz, a ne vrednost.** To je jedina suptilnost u metodi. Prvi zadatak sme da dobije
**više** argumenata, svaki sledeći **tačno jedan**. Da bi oba prošla kroz isti red, kroz petlju se nosi
**niz argumenata** i razlaže sa `...ulaz`. Otud i `ulaz[0]` na kraju.

In [ ]:
console.log("Z9 lancano:", await ravan.execute("1,0,2,3"));
console.log("   '1,0,2,3' -> [1,0,2,3] -> [1,2,3] -> [1,2,3] -> 6 -> tekst");

console.log("\nZ10 prvi dobija tri argumenta:", await new Workflow("t", "B", [saberiTri, puta10]).execute(2, 3, 5));
console.log("   (2+3+5) * 10 = 100 — puta10 je dobio samo rezultat, jedan argument");

console.log("\nZ11 vraca se poslednji:", await new Workflow("dva", "B", [parsiraj, zbir]).execute("1,2,3"));
console.log("   parsiraj bi dao [1,2,3], ali se vraca 6 — sto je izlaz zbira");

console.log("\nZ12 Task.execute vraca Promise:  ", zbir.execute([1]) instanceof Promise);
console.log("Z12 Workflow.execute vraca Promise:", ravan.execute("1") instanceof Promise);

// Z9 jos jednom: da se stvarno CEKA, a ne izvrsava uporedo
const trag = [];
const spor = new Task("spor", async (x) => {
  await new Promise((k) => setTimeout(k, 30));
  trag.push("spor");
  return x;
});
const brz = new Task("brz", (x) => { trag.push("brz"); return x; });
await new Workflow("redosled", "B", [spor, brz]).execute(1);
console.log("\nZ9 redosled:", trag, "— brz je cekao spor iako je brz");

---
# Pasus 3 — transdjuseri

> „**Modifikovati Workflow objekat tako da je nad njim moguće primeniti proizvoljnu kompoziciju
> transdjusera.** Demonstrirati ispravnost implementacije **definisanjem transdjusera za ispis svakog
> koraka izvršavanja** Workflow objekta."

## Z13 — proizvoljna kompozicija

Traži se **modifikacija Workflow objekta**, pa su dodate dve metode:

```js
this.transduce = function (transdjuser, kraj = sakupi, pocetno = []) {   // ishod bilo kakav
  return zadaci.reduce(transdjuser(kraj), pocetno);
};

this.primeni = function (...transdjuseri) {                              // ishod je opet Workflow
  return new Workflow(naziv, autor, zadaci.reduce(poredjaj(...transdjuseri)(sakupi), []));
};
```

Reč **proizvoljna** je pokrivena sa `...transdjuseri` — koliko god ih se prosledi, `poredjaj` ih spaja.

Transdjuser je funkcija koja **prima redjuser i vraća redjuser**:

| Deo | Uloga |
| --- | --- |
| `sakupi` | krajnji redjuser — odlučuje **šta je ishod** |
| `preslikaj`, `izdvoj` | transdjuseri — odlučuju **šta se usput radi** |
| `poredjaj` | spaja proizvoljno mnogo transdjusera u jedan |

**Bitno:** kroz transdjuser prolaze **zadaci**, ne podaci koje zadaci obrađuju.

## Z14 — transdjuser za ispis svakog koraka

Traženi transdjuser ne traži novu mašineriju — to je običan `preslikaj` koji svaki zadatak zamenjuje
**novim zadatkom** koji oko izvršavanja dopisuje ispis:

```js
const saIspisom = preslikaj(function (zadatak) {
  return new Task(zadatak.naziv, async function (...argumenti) {
    console.log("   ->", zadatak.naziv, "ulaz:", ...argumenti);
    const rezultat = await zadatak.execute(...argumenti);   // original radi svoj posao
    console.log("   <-", zadatak.naziv, "izlaz:", rezultat);
    return rezultat;                                        // isti rezultat kao bez sloja
  });
});
```

In [ ]:
const sakupi = function (niz, element) { return [...niz, element]; };      // krajnji redjuser
const prebroj = function (broj) { return broj + 1; };                      // drugi krajnji redjuser

const preslikaj = function (posao) {                                       // Z13 transdjuser
  return function (sledeci) {
    return function (stanje, element) {
      return sledeci(stanje, posao(element));
    };
  };
};

const izdvoj = function (provera) {                                        // Z13 transdjuser
  return function (sledeci) {
    return function (stanje, element) {
      return provera(element) ? sledeci(stanje, element) : stanje;
    };
  };
};

const poredjaj = function (...transdjuseri) {                              // Z13 kompozicija
  return function (kraj) {
    let veza = kraj;
    for (let i = transdjuseri.length - 1; i >= 0; i--) veza = transdjuseri[i](veza);
    return veza;
  };
};

const saIspisom = preslikaj(function (zadatak) {                           // Z14
  return new Task(zadatak.naziv, async function (...argumenti) {
    console.log("   ->", zadatak.naziv, "ulaz:", ...argumenti);
    const rezultat = await zadatak.execute(...argumenti);
    console.log("   <-", zadatak.naziv, "izlaz:", rezultat);
    return rezultat;
  });
});

// ── dokazi ────────────────────────────────────────────────────────
console.log("Z13 jedan transdjuser:", ravan.transduce(preslikaj((z) => z.naziv)));
console.log("Z13 drugi krajnji redjuser:", ravan.transduce(preslikaj((z) => z), prebroj, 0), "zadataka");
console.log("Z13 kompozicija dva:", ravan.transduce(poredjaj(izdvoj((z) => z.naziv !== "cekaj"), preslikaj((z) => z.naziv))));
console.log("Z13 primeni vraca Workflow:", ravan.primeni(izdvoj((z) => z.naziv !== "cekaj")).zadaci.length, "zadatka");

console.log("\nZ14 ispis svakog koraka:");
console.log("   rezultat:", await ravan.primeni(saIspisom).execute("1,0,2,3"));

console.log("\nZ13 proizvoljna kompozicija — dva transdjusera zajedno:");
console.log("   rezultat:", await ravan.primeni(izdvoj((z) => z.naziv !== "cekaj"), saIspisom).execute("4,0,5"));

console.log("\nZ14 original nije obmotan:", ravan.zadaci[0] === parsiraj);

---
# Pasus 4 — demonstracija

> „Demonstrirati ispravnost rešenja **instanciranjem nekoliko Workflow i Task objekata** i
> **pozivanjem metoda definisanih nad njima**."

| | Zahtev | Čime |
| --- | --- | --- |
| **Z15** | nekoliko Task objekata | `parsiraj`, `bezNula`, `zbir`, `formatiraj`, `cekaj`, `saberiTri`, `puta10` |
| **Z15** | nekoliko Workflow objekata | `priprema`, `racun`, `glavni`, `ravan` |
| **Z15** | pozvane metode | `execute`, `map`, `filter`, `flat`, `flatMap`, `transduce`, `primeni` |

In [ ]:
console.log("Z15 Task objekata:", [parsiraj, bezNula, zbir, formatiraj, cekaj, saberiTri, puta10].length);
console.log("Z15 Workflow objekata:", [priprema, racun, glavni, ravan].length);

console.log("\nZ15 pozvane metode nad Workflow-om:");
console.log("   map:      ", [...ravan.map((z) => z)].length, "zadataka");
console.log("   filter:   ", [...ravan.filter((z) => z.naziv !== "cekaj")].length, "zadataka");
console.log("   flat:     ", [...glavni.flat()].length, "zadataka");
console.log("   flatMap:  ", [...priprema.flatMap((z) => new Workflow("p", "B", [z]))].length, "zadataka");
console.log("   transduce:", ravan.transduce(preslikaj((z) => z.naziv)).length, "naziva");
console.log("   primeni:  ", ravan.primeni(saIspisom).zadaci.length, "obmotanih");
console.log("   execute:  ", await ravan.execute("1,0,2,3"));

console.log("\nZ15 pozvane metode nad Task-om:");
console.log("   execute jedan argument:", await parsiraj.execute("1,2"));
console.log("   execute tri argumenta: ", await saberiTri.execute(1, 2, 3));

console.log("\nZ15 sve i dalje radi posle svih operacija:", [...ravan].map((z) => z.naziv));

---
# Spisak za proveru

| | Zahtev | Gde u kodu | Dokaz u ispisu |
| --- | --- | --- | --- |
| Z1 | konstruktori Workflow i Task | `const Task = function`, `const Workflow = function` | instance postoje |
| Z2 | naziv, definicija, execute | tri reda u `Task` | `Z2 tri polja` |
| Z3 | definicija se izvršava pozivom execute | `return definicija(...argumenti)` | `Z3 definicija je funkcija` |
| Z4 | rezultat = rezultat definicije | isti red | `Z4 execute = poziv definicije` |
| Z5 | proizvoljan broj argumenata | `(...argumenti)` / `definicija(...argumenti)` | `Z5 jedan argument … tri` |
| Z6 | naziv, autor, spisak, execute | `this.naziv`, `this.autor`, geter `zadaci`, `this.execute` | `Z6 podaci`, `Z6 spisak zasticen` |
| Z7 | iterabilan, daje Task objekte | `this[Symbol.iterator] = function*` | `Z7 for...of`, `Z7 spread` |
| Z8 | map, flat, flatMap, filter | četiri metode | `Z8 flat/filter/map/flatMap` |
| Z9 | ulaz sledećeg = rezultat prethodnog | `ulaz = [await zadatak.execute(...ulaz)]` | `Z9 lancano`, `Z9 redosled` |
| Z10 | prvi dobija sve argumente | `let ulaz = argumenti` | `Z10 prvi dobija tri argumenta` |
| Z11 | vraća se rezultat poslednjeg | `return ulaz[0]` | `Z11 vraca se poslednji` |
| Z12 | oba execute asinhrona | `async` na obe metode | `Z12 … vraca Promise` |
| Z13 | proizvoljna kompozicija transdjusera | `transduce`, `primeni`, `poredjaj` | `Z13 kompozicija dva` |
| Z14 | transdjuser za ispis svakog koraka | `saIspisom` | `Z14 ispis svakog koraka` |
| Z15 | demonstracija | poslednja ćelija | `Z15 …` |

## Tri mesta na koja vredi skrenuti pažnju

**Z10 je najčešće promašen.** Većina rešenja prvom zadatku prosledi samo jedan argument. Zato akumulator
nosi **niz** argumenata, a ne vrednost — i zato na kraju stoji `ulaz[0]`.

**Z7 nije polje nego sposobnost.** Zadatak kaže „Workflow **je** iterabilni objekat", ne „sadrži niz".
Zbog `Symbol.iterator` rade `for…of`, spread, `Array.from` i razlaganje — i zbog toga `flat` može da
raspakuje ugnježđeni Workflow ne pitajući ga za tip.

**Z14 nije posebna mašinerija.** Traženi transdjuser je obična upotreba `preslikaj`-a. Da je pisan kao
zaseban mehanizam, ne bi se mogao slagati sa ostalima — a Z13 traži baš to.